In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.distributions as distributions
import numpy as np
import gymnasium as gym 

# Aprendizagem por reforço com o *Gymnasium*: Um guia prático

## Compreenda os conceitos básicos da Aprendizagem por Reforço (RL) e explore o pacote de software *Gymnasium* para construir e testar algoritmos de RL utilizando Python.


A Aprendizagem por Reforço (AR) é um dos três principais paradigmas da aprendizagem automática, sendo os outros dois a aprendizagem supervisionada e a não supervisionada. Na AR, um agente aprende a interagir com o seu ambiente para maximizar as recompensas acumuladas. Aprende a ação óptima em diferentes condições ambientais através de tentativa e erro.

A AR resolve problemas como carros autónomos, comércio automatizado, jogadores de computador em jogos de vídeo, robôs de treino e muito mais. Quando são utilizadas redes neuronais profundas para aplicar algoritmos de AR, chama-se Aprendizagem por Reforço Profunda.

Neste tutorial, vou mostrar-lhe como começar a utilizar o *Gymnasium*, uma biblioteca Python de código aberto para desenvolver e comparar algoritmos de aprendizagem por reforço. Vou demonstrar como configurá-la, explorar vários ambientes de AR e usar Python para construir um agente simples para implementar um algoritmo de AR.

# O que é o *Gymnasium*?

O *Gymnasium* é uma biblioteca Python de código aberto concebida para apoiar o desenvolvimento de algoritmos de AR. Para facilitar a pesquisa e o desenvolvimento em AR, o *Gymnasium* fornece:

- Uma grande variedade de ambientes, desde jogos simples a problemas que imitam cenários reais.

- APIs e *wrappers* simplificados para interagir com os ambientes.

- A capacidade de criar ambientes personalizados e tirar partido da estrutura da API.

Os programadores podem criar algoritmos AR e utilizar chamadas API para tarefas como:

- Transmitir ao ambiente a ação escolhida pelo agente.

- Conhecer o estado do ambiente e a recompensa após cada ação.

- Treinar o modelo.

- Testar o desempenho do modelo.

# O ginásio da OpenAI versus o ginásio do Farama

A OpenAI não afectou recursos significativos ao desenvolvimento do *Gym* porque não era uma prioridade comercial para a empresa. A [Fundação Farama](https://farama.org/Announcing-The-Farama-Foundation)  foi criada para padronizar e manter as bibliotecas AR a longo prazo. *Gymnasium* é o fork da Fundação Farama para o *Gym* da OpenAI. *Gymnasium* 0.26.2 é um substituto para o *Gym* 0.26.2. Com o fork, a Farama pretende adicionar métodos funcionais (além dos baseados em classes) para todas as chamadas de API, suportar ambientes vetoriais e melhorar os wrappers. O objetivo geral é tornar o framework mais limpo e mais eficiente.

# Instalação do ginásio

O *Gymnasium* precisa de versões específicas (não as últimas versões) de vários programas de dependência como o NumPy e o PyTorch. Assim, recomendamos que crie um novo ambiente Conda ou venv ou um novo notebook para instalar, usar o *Gymnasium* e executar os programas AR.

## Instalação do ginásio

Para instalar o *Gymnasium* num servidor ou numa máquina local, execute:


In [ ]:
$ pip install gymnasium 

Para instalar utilizando um Notebook faça:

In [ ]:
!pip install gymnasium

A partir de novembro de 2024, o *Gymnasium* inclui mais de 60 ambientes incorporados. Para pesquisar os ambientes integrados disponíveis, utilize a função `gym.envs.registry.all()`, como ilustrado no exemplo abaixo:

In [ ]:
import gymnasium as gym
for i in gym.envs.registry.keys():
	print(i)

Também pode visitar a página inicial do [*Gymnasium*](https://gymnasium.farama.org/). A coluna da esquerda contém ligações para todos os ambientes. A página Web de cada ambiente inclui pormenores sobre ele, como acções, estados, etc.

Os ambientes estão organizados em categorias como Controlo Clássico, Box2D, entre outras. Abaixo, listo alguns dos ambientes comuns em cada grupo:

`Controlo clássico`: Estes são os ambientes canónicos utilizados no desenvolvimento de AR; constituem a base de muitos exemplos de livros didácticos. Eles oferecem a combinação certa de complexidade e simplicidade para testar e avaliar novos algoritmos de RL. Os ambientes de controlo clássicos no *Gymnasium* incluem:

- Acrobata

- Poste do carrinho

- Carro de montanha discreto

- Carro de montanha contínuo

- Pêndulo

`Box2D`: Box2D é um motor de física 2D para jogos. Os ambientes baseados neste motor incluem jogos simples como:

- Lunar Lander

- Corridas de carros

`ToyText`: Trata-se de ambientes pequenos e simples, frequentemente utilizados para depurar algoritmos de AR. Muitos destes ambientes baseiam-se no modelo de mundo de grelha pequena e em jogos de cartas simples. Os exemplos incluem:

- Blackjack

- Táxi

- Lago congelado

`MuJoCo`: O MuJoCo (Multi-Joint dynamics with Contact) é um motor de física de código aberto que simula ambientes para aplicações como a robótica, a biomecânica, o ML, etc. Os ambientes MuJoCo no *Gymnasium* incluem:

- Formiga

- Tremonha

- Humanoide

- Nadador

E mais

Para além dos ambientes integrados, o *Gymnasium* pode ser utilizado com muitos ambientes externos utilizando a mesma API.

Iremos utilizar um dos ambientes canónicos do `Controlo Clássico` (*Classic Control*) neste tutorial. Para importar um ambiente específico, use o comando `.make()` e passe o nome do ambiente como argumento. Por exemplo, para criar um novo ambiente baseado no `CartPole (versão 1)`, use o comando abaixo:

In [ ]:
import gymnasium as gym
env = gym.make("CartPole-v1")

# Compreender os conceitos de aprendizagem por reforço no ginásio

Em poucas palavras, a Aprendizagem por Reforço consiste num agente (como um robot) que interage com o seu ambiente. Uma política decide as acções do agente. Em função das acções do agente, o ambiente atribui-lhe uma recompensa (ou uma penalização) em cada passo de tempo. O agente usa a AR para descobrir a política óptima que maximiza as recompensas totais que o agente ganha.

## Componentes de um ambiente de AR

Os componentes principais de um ambiente de AR são os seguintes

- Ambiente: O sistema externo, mundo ou contexto. O agente interage com o ambiente numa série de passos temporais. Em cada passo de tempo, com base na ação do agente, o ambiente:

	- Dá uma recompensa (ou penalização)

	- Decide o estado seguinte

- Estado: Uma representação matemática da configuração atual do ambiente.

	- Por exemplo, o estado de um ambiente de pêndulo pode incluir a posição e a velocidade angular do pêndulo em cada passo de tempo.

	- Estado terminal: Um estado que não conduz a novos/outros estados.

- Agente: O algoritmo que observa o ambiente e toma várias acções com base nessa observação. O objetivo do agente é maximizar as suas recompensas.

	- Por exemplo, o agente decide com que força e em que direção deve empurrar o pêndulo.  

- Observação: Uma representação matemática da visão que o agente tem do ambiente, adquirida, por exemplo, através de sensores.

- Ação: A decisão tomada pelo agente antes de avançar para o passo seguinte. A ação afecta o estado seguinte do ambiente e dá ao agente uma recompensa.

- Recompensa: O feedback do ambiente para o agente. Pode ser positiva ou negativa, dependendo da ação e do estado do ambiente.

- Retorno: O retorno cumulativo esperado ao longo de passos de tempo futuros. As recompensas de passos de tempo futuros podem ser descontadas utilizando um fator de desconto.

- Política: A estratégia do agente sobre qual a ação a tomar em vários estados. É tipicamente representada como uma matriz de probabilidade, P, que mapeia estados para acções.

	- Dado um conjunto finito de m estados possíveis e n acções possíveis, o elemento Pmn na matriz denota a probabilidade de tomar a ação an no estado sm.  

- Episódio: A série de passos temporais desde o estado inicial (aleatório) até o agente atingir um estado terminal.

## Espaço de observação e espaço de ação

A observação é a informação que o agente recolhe sobre o ambiente. Um agente, por exemplo, um robot, pode recolher informações sobre o ambiente utilizando sensores. Idealmente, o agente deve ser capaz de observar o estado completo, que descreve todos os aspectos do ambiente. Na prática, o agente utiliza as suas observações como um substituto para o estado. Assim, as observações decidem as acções do agente.

Um espaço é análogo a um conjunto matemático. O espaço dos objectos X inclui todas as instâncias possíveis de X. O espaço de X define igualmente a estrutura (sintaxe e formato) de todos os objectos do tipo X. Cada ambiente *Gymnasium* tem dois espaços, o espaço de ação, `action_space`, e o espaço de observação, `observation_space`. Tanto o espaço de ação como o espaço de observação derivam da superclasse pai `gymnasium.spaces.Space`.

### Espaço de observação

O espaço de observação é o espaço que inclui todas as observações possíveis. Define também o formato em que as observações são armazenadas. O espaço de observação é normalmente representado como um objeto do tipo de dados `Box`. Trata-se de um `ndarray` que descreve os parâmetros das observações. A caixa especifica os limites de cada dimensão. Pode ver o espaço de observação de um ambiente utilizando o método `observation_space`:

In [ ]:
print("observation space: ", env.observation_space)

No caso do ambiente CartPole-v1, a saída é semelhante ao exemplo abaixo:

In [ ]:
observation space:  Box([-4.8 -inf -0.41887903 -inf], [4.8 inf 0.41887903 inf], (4,), float32)

Neste exemplo, o espaço de observação do `CartPole-v1` tem 4 dimensões. Os 4 elementos da matriz de observação são:

- Posição do carrinho - varia entre -4,8 e +4,8

- Velocidade do carro - varia entre -$\infty$ e +$\infty$

- Ângulo do pólo - varia entre -0,4189 = rad(-24º) e +0,4189 = rad(24º)

- Velocidade angular do pólo - varia entre -$\infty$ e +$\infty$

Para ver um exemplo de uma matriz de observação individual, utilize o comando `.reset()`.

In [ ]:
observation, info = env.reset()
print("observation: ", observation)

No caso do ambiente `CartPole-v1`, a saída é semelhante ao exemplo abaixo:

In [ ]:
[ 0.03481963 -0.0277232   0.01703267 -0.04870504]                                                                                                       

Os quatro elementos desta matriz correspondem às quatro grandezas observadas (posição do carrinho, velocidade do carrinho, ângulo do pólo, velocidade angular do pólo), como explicado anteriormente.

### Espaço de ação

O espaço de ação inclui todas as acções possíveis que o agente pode realizar. O espaço de ação também define o formato em que as acções são representadas. Pode visualizar o espaço de ação de um ambiente utilizando o método `action_space`:

In [ ]:
print("action space: ", env.action_space)

No caso do ambiente CartPole-v1, a saída é semelhante ao exemplo abaixo:

In [ ]:
action space:  Discrete(2)

No caso do ambiente `CartPole-v1`, o espaço de ação é discreto. Há um total de duas acções que o agente pode realizar:

0: Empurra o carrinho para a esquerda

1: Empurra o carrinho para a direita

# Criar o seu primeiro agente RL com o *Gymnasium*

Nas secções anteriores, explorámos os conceitos básicos da AR e do *Gymnasium*. Esta secção mostra-lhe como utilizar o *Gymnasium* para construir um agente AR.

## Criando e redefinindo o ambiente

O primeiro passo é criar uma instância do ambiente. Para criar novos ambientes, utilize o método `.make()`.

In [ ]:
env = gym.make('CartPole-v1')

As interações do agente alteram o estado do ambiente. O método `.reset()` redefine o ambiente para um estado inicial. Por padrão, o ambiente é inicializado em um estado aleatório. Você pode usar um parâmetro SEED com o método `.reset()` para inicializar o ambiente no mesmo estado sempre que o programa for executado. O código abaixo mostra como fazer isso:

In [ ]:
SEED = 1111
env.reset(seed=SEED)

A amostragem de acções também envolve aleatoriedade. Para controlar esta aleatoriedade e obter um percurso de treino totalmente reproduzível, podemos semear os geradores aleatórios do NumPy e do PyTorch:

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

## Acções aleatórias versus acções inteligentes

Em cada etapa de um processo de Markov, o agente pode escolher aleatoriamente uma ação e explorar o ambiente até chegar a um estado terminal. Ao escolher acções ao acaso:

- Pode demorar muito tempo a atingir o estado terminal.

- As recompensas acumuladas são muito inferiores ao que poderiam ter sido.

Treinar o agente para otimizar a seleção de acções com base em experiências anteriores (de interação com o ambiente) é mais eficiente para maximizar as recompensas a longo prazo.

O agente não treinado começa com acções aleatórias baseadas numa política inicializada aleatoriamente. Esta política é normalmente representada como uma rede neural. Durante o treino, o agente aprende a política óptima que maximiza as recompensas. Em AR, o processo de treino é também designado por otimização de políticas.

Existem vários métodos de otimização de políticas. As `equações de Bellman` descrevem como calcular o valor das políticas em AR e determinar a política óptima. Neste tutorial, usaremos uma técnica simples chamada gradientes de política (*policy gradients*). Existem outros métodos, como a Otimização de política proximal (PPO) e outros.

# Implementação de um agente de gradiente de política simples

Para construir um agente de AR que utilize gradientes de política, criamos uma rede neural para implementar a política, escrevemos funções para calcular os retornos e as perdas a partir das recompensas por etapas e das probabilidades de ação, e actualizamos iterativamente a política utilizando técnicas padrão de retropropagação.

## Configurar a rede de políticas

Utilizamos uma rede neural para implementar a política. Como o `CartPole-v1` é um ambiente simples, utilizamos uma rede neural com:

- Dimensões de entrada iguais à dimensionalidade do espaço de observação do ambiente.

- Uma única camada oculta com 64 neurónios.

- Dimensões de saída iguais à dimensionalidade do espaço de ação do ambiente.

Assim, a função da rede de políticas é mapear os estados observados para as acções. Dada uma observação de entrada, prevê a ação correta. O código abaixo implementa a rede de políticas:

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = self.layer1(x)
        x = self.dropout(x)
        x = F.relu(x)
        x = self.layer2(x)
        return x

## A recolha de recompensas e o passe para a frente

Como referido, em cada etapa do processo de Markov, o ambiente atribui uma recompensa com base na ação e no estado do agente. O objetivo da AR é maximizar o retorno total.

- O retorno em cada passo de tempo é a soma acumulada das recompensas obtidas desde o início até esse passo.

- O retorno total em cada episódio é obtido através da acumulação de todas as recompensas por etapas desse episódio. Assim, o retorno total é o retorno no último passo de tempo (quando o agente atinge um estado terminal).

Na prática, ao acumular prémios, é comum:

- Ajuste as recompensas futuras utilizando um fator de desconto.

- Normalize a matriz de retornos por etapas para garantir uma formação suave e estável.

O código abaixo mostra como o pode fazer:

In [ ]:
def calculate_stepwise_returns(rewards, discount_factor):
    returns = []
    R = 0
    for r in reversed(rewards):
        R = r + R * discount_factor
        returns.insert(0, R)
    returns = torch.tensor(returns)
    normalized_returns = (returns - returns.mean()) / returns.std()
    return normalized_returns

A passagem para a frente consiste em executar o agente com base na política atual até atingir um estado terminal e recolher as recompensas por etapas e as probabilidades de ação. Os passos seguintes explicam como implementar a passagem progressiva:

- Reponha o ambiente num estado inicial.

- Inicialize os buffers para armazenar as probabilidades de ação, as recompensas e o retorno acumulado

- Utilize a função .step() para executar iterativamente o agente no ambiente até terminar:

	- Obtenha a observação do estado do ambiente.

	- Obtenha a ação prevista pela política com base na observação.

	- Utilize a função `Softmax` para estimar a probabilidade de realizar a ação prevista.

	- Simule uma distribuição de probabilidade categórica com base nestas probabilidades estimadas.

	- Recolha amostras desta distribuição para obter a ação do agente.

	- Estime a probabilidade logarítmica da ação amostrada a partir da distribuição simulada.

- Anexe a probabilidade logarítmica das acções e as recompensas de cada etapa aos seus respectivos buffers.

- Estime os valores normalizados e descontados dos retornos em cada etapa com base nas recompensas.

In [ ]:
def forward_pass(env, policy, discount_factor):
    log_prob_actions = []
    rewards = []
    done = False
    episode_return = 0
    policy.train()
    observation, info = env.reset()
    while not done:
        observation = torch.FloatTensor(observation).unsqueeze(0)
        action_pred = policy(observation)
        action_prob = F.softmax(action_pred, dim = -1)
        dist = distributions.Categorical(action_prob)
        action = dist.sample()
        log_prob_action = dist.log_prob(action)
        observation, reward, terminated, truncated, info = env.step(action.item())
        done = terminated or truncated
        log_prob_actions.append(log_prob_action)
        rewards.append(reward)
        episode_return += reward
    log_prob_actions = torch.cat(log_prob_actions)
    stepwise_returns = calculate_stepwise_returns(rewards, discount_factor)
    return episode_return, stepwise_returns, log_prob_actions

## Atualização da política com base nas recompensas

A perda representa a quantidade à qual aplicamos a descida do gradiente (*gradient descent*). O objetivo da AR é maximizar os retornos. Por isso, usamos o valor de retorno esperado como um substituto para a perda. O valor de retorno esperado é calculado como o produto dos retornos esperados passo a passo e a probabilidade logarítmica das acções passo a passo. O código abaixo calcula a perda:

In [ ]:
def calculate_loss(stepwise_returns, log_prob_actions):
    loss = -(stepwise_returns * log_prob_actions).sum()
    return loss

Para atualizar a política, execute a retropropagação em relação à função de perda. O método `update_policy()` abaixo invoca o método `calculate_loss()`. Em seguida, executa a retropropagação sobre essa perda para atualizar os parâmetros da política, ou seja, os pesos do modelo da rede neuronal de políticas.

In [ ]:
def update_policy(stepwise_returns, log_prob_actions, optimizer):
    stepwise_returns = stepwise_returns.detach()
    loss = calculate_loss(stepwise_returns, log_prob_actions)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

A atualização da política com base no gradiente dos retornos é designada por método do gradiente da política (*policy gradient*).

## Formação da política

Dispomos agora de todos os componentes necessários para formar e avaliar a política. Implementamos o ciclo de formação como explicado nos passos seguintes:  

Antes de começar, declaramos os hiperparâmetros, instanciamos uma política e criamos um optimizador:

- Declare os hiperparâmetros como constantes Python:

	- `MAX_EPOCHS` é o número máximo de iterações que estamos dispostos a executar para treinar a política.

	- ``DISCOUNT_FACTOR` decide a importância relativa das recompensas de passos de tempo futuros. Um fator de desconto de 1 significa que todas as recompensas são igualmente importantes, enquanto um valor de 0 significa que apenas a recompensa do passo de tempo atual é importante.

	- `N_TRIALS` é o número de episódios sobre os quais calculamos a média dos retornos para avaliar o desempenho do agente. Decidimos que a formação é bem sucedida se o retorno médio dos episódios `N_TRIALS` for superior ao limiar.

	- `REWARD_THRESHOLD`: Se a política conseguir obter um retorno superior ao limiar, é considerada bem sucedida.

	- `DROPOUT` decide a fração dos pesos que deve ser aleatoriamente zerada. A função dropout define aleatoriamente uma fração dos pesos do modelo como zero. Isso reduz a dependência de neurônios específicos e evita o ajuste excessivo, tornando a rede mais robusta.

	- `LEARNING_RATE` decide em que medida os parâmetros da política podem ser modificados em cada passo. A atualização dos parâmetros em cada iteração é o produto do gradiente e da taxa de aprendizagem.

- Defina a política como uma instância da classe PolicyNetwork (implementada anteriormente).

- Crie um optimizador utilizando o algoritmo `Adam` e a taxa de aprendizagem.

Para treinar a política, executamos iterativamente os passos de treino até que o retorno médio (sobre N_TRIALS) seja superior ao limiar de recompensa:

- Para cada episódio, execute a passagem para a frente uma vez. Recolha o logaritmo da probabilidade das acções, os retornos por etapas e o retorno total desse episódio. Acumule os retornos episódicos numa matriz.

- Calcule a perda utilizando as probabilidades logarítmicas e os retornos por etapas. Execute a retropropagação na perda. Utilize o optimizador para atualizar os parâmetros da política.

- Verifique se o retorno médio em `N_TRIALS` excede o limiar de recompensa.

O código abaixo implementa estes passos:

In [ ]:
def main(): 
    MAX_EPOCHS = 500
    DISCOUNT_FACTOR = 0.99
    N_TRIALS = 25
    REWARD_THRESHOLD = 475
    PRINT_INTERVAL = 10
    INPUT_DIM = env.observation_space.shape[0]
    HIDDEN_DIM = 128
    OUTPUT_DIM = env.action_space.n
    DROPOUT = 0.5
    episode_returns = []
    policy = PolicyNetwork(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM, DROPOUT)
    LEARNING_RATE = 0.01
    optimizer = optim.Adam(policy.parameters(), lr = LEARNING_RATE)
    for episode in range(1, MAX_EPOCHS+1):
        episode_return, stepwise_returns, log_prob_actions = forward_pass(env, policy, DISCOUNT_FACTOR)
        _ = update_policy(stepwise_returns, log_prob_actions, optimizer)
        episode_returns.append(episode_return)
        mean_episode_return = np.mean(episode_returns[-N_TRIALS:])
        if episode % PRINT_INTERVAL == 0:
            print(f'| Episode: {episode:3} | Mean Rewards: {mean_episode_return:5.1f} |')
        if mean_episode_return >= REWARD_THRESHOLD:
            print(f'Reached reward threshold in {episode} episodes')
            break

Por fim, invoque a função `main()` para treinar a política:


In [ ]:
main()

# Técnicas avançadas de *Gymnasium*

Tendo demonstrado como implementar um algoritmo de AR, passamos agora a discutir algumas técnicas avançadas normalmente utilizadas na prática.

## Utilizar arquitecturas pré-construídas

A implementação de algoritmos de AR a partir do zero é um processo longo e difícil, especialmente para ambientes complexos e políticas de ponta.

Uma alternativa mais prática é utilizar software como o `Stable Baselines3`. Este software inclui implementações experimentadas e testadas de algoritmos de AR. Inclui agentes pré-treinados, scripts de treino, ferramentas de avaliação e módulos para traçar gráficos e gravar vídeos.

`Ray RLib` é outra ferramenta popular para AR. O `RLib` foi concebido como uma solução escalável, facilitando a implementação de algoritmos de AR em sistemas multi-GPU. Também suporta AR multi-agente, o que abre novas possibilidades como:

- Aprendizagem multiagente independente: Cada agente trata os outros agentes como parte do ambiente.

- Formação multiagente colaborativa: Um grupo de agentes partilha a mesma política e funções de valor e aprende com as experiências uns dos outros em paralelo.

- Formação adversarial: Os agentes (ou grupos de agentes) competem uns contra os outros em ambientes competitivos semelhantes a jogos.

Tanto com o `RLib` como com o `Stable Baselines3`, se pode importar e utilizar ambientes do OpenAI *Gymnasium*.

## Ambientes personalizados

Os ambientes que acompanham o *Gymnasium* são a escolha certa para testar novas estratégias de AR e políticas de formação. No entanto, para a maioria das aplicações práticas, é necessário criar e utilizar um ambiente que reflicta com precisão o problema que pretende resolver. Pode utilizar o *Gymnasium* para criar um ambiente personalizado. A vantagem de utilizar os ambientes personalizados *Gymnasium* é que muitas ferramentas externas, como RLib e Stable Baselines3, já estão configuradas para trabalhar com a estrutura da API *Gymnasium*.

Para criar um ambiente personalizado no *Gymnasium*, tem de definir:

- O espaço de observação.

- As condições terminais.

- O conjunto de acções que o agente pode escolher.

- Como inicializar o ambiente (quando a função `reset()` é chamada).

- A forma como o ambiente decide o estado seguinte, tendo em conta as acções do agente (quando a função `step()` é chamada).


## Melhores práticas de utilização do *Gymnasium*

### Experimente com diferentes ambientes

O código neste tutorial mostrou como implementar o algoritmo de gradiente de política no ambiente `CartPole`. Este é um ambiente simples com um espaço de ação discreto. Para compreender melhor a AR, aconselhamo-lo a aplicar o mesmo algoritmo de gradiente de política (e outros algoritmos, como o `PPO`) noutros ambientes.

Por exemplo, o ambiente Pendulum tem um espaço de ação contínuo. É constituído por uma única entrada representada como uma variável contínua - o binário (magnitude e direção do) aplicado ao pêndulo num determinado estado. Este binário pode assumir qualquer valor entre -2 e +2.

A experimentação de diferentes algoritmos em vários ambientes ajuda-o a compreender melhor os diferentes tipos de soluções de AR e os seus desafios.

### Monitorize o progresso da formação

Os ambientes AR são frequentemente constituídos por robôs, pêndulos, carros de montanha, jogos de vídeo, etc. A visualização das acções do agente no ambiente permite uma melhor compreensão intuitiva do desempenho da política.

No *Gymnasium*, o método `env.render()` visualiza as interações do agente com o ambiente. Apresenta graficamente o estado atual do ambiente - ecrãs de jogo, posição do pêndulo ou do poste do carrinho, etc. O feedback visual das acções do agente e das respostas do ambiente ajuda a monitorizar o desempenho do agente e o seu progresso ao longo do processo de formação.

Existem quatro modos de renderização: "`human`", "`rgb_array`", "`ansi`" e "`rgb_array_list`". Para visualizar o desempenho do agente, use o modo de renderização "`human`". O modo de renderização é especificado quando o ambiente é inicializado. Por exemplo:

In [ ]:
env = gym.make("CartPole-v1", render_mode="human")

Para efetuar a renderização, envolva o método `.render()` após cada ação executada pelo agente (através da chamada do método `.step()`). O pseudo-código abaixo ilustra como fazer isso:


In [ ]:
while not done:
   step, reward, terminated, truncated, info = env.step(action.item())
   env.render()


## Resolução de problemas de erros comuns

O *Gymnasium* facilita a interface com ambientes AR complexos. No entanto, é um software em constante atualização e com muitas dependências. Por isso, é essencial estar atento a alguns tipos de erros comuns.

### Incompatibilidades de versões

- Incompatibilidade de versão do *Gymnasium*: O pacote de software *Gymnasium* de Farama foi bifurcado do Gym da OpenAI a partir da versão 0.26.2. Houve algumas mudanças significativas entre as versões antigas do Gym e as novas versões do *Gymnasium*. Muitas implementações disponíveis publicamente são baseadas nas versões mais antigas do Gym e podem não funcionar diretamente com a última versão. Nesses casos, é necessário reverter a instalação para uma versão anterior ou adaptar o código para funcionar com a versão mais recente.

- Incompatibilidade da versão do ambiente: Muitos ambientes do *Gymnasium* têm versões diferentes. Por exemplo, existem dois ambientes CartPole - CartPole-v1 e CartPole-v0. Embora o comportamento do ambiente seja o mesmo em ambas as versões, alguns dos parâmetros, como a duração do episódio, o limiar de recompensa, etc., podem ser diferentes. Uma política treinada numa versão pode não ter um desempenho tão bom noutra versão do mesmo ambiente. Tem de atualizar os parâmetros de treino e voltar a treinar a política para cada versão do ambiente.

- Incompatibilidade de versões de dependências: O *Gymnasium* depende de dependências como NumPy e PyTorch. Em dezembro de 2024, as últimas versões dessas dependências são numpy 2.1.3 e torch 2.5.1. No entanto, o *Gymnasium* funciona melhor com torch 1.13.0 e numpy 1.23.3. Você pode encontrar problemas se instalar o *Gymnasium* num ambiente com essas dependências pré-instaladas. Recomendamos que instale e trabalhe com o *Gymnasium* num ambiente Conda novo.

### Problemas de convergência

- Hiperparâmetros: Tal como outros algoritmos de aprendizagem automática, as políticas de AR são sensíveis a hiperparâmetros como a taxa de aprendizagem, o fator de desconto, etc. Recomendamos que experimente e ajuste os hiperparâmetros manualmente ou utilizando técnicas automatizadas como a pesquisa em grelha e a pesquisa aleatória.

- Exploração versus exploração: Para algumas classes de políticas (como a `PPO`), o agente adopta uma estratégia com duas vertentes: explorar o ambiente para descobrir novos caminhos e adotar uma abordagem gulosa para maximizar as recompensas com base nos caminhos conhecidos até ao momento. Se explorar demasiado, a política não converge. Por outro lado, nunca tenta o caminho ótimo se não explorar o suficiente. Por isso, é essencial encontrar o equilíbrio correto entre exploração e aproveitamento. Também é comum dar prioridade à exploração nos primeiros episódios e à exploração nos últimos episódios durante o treino.

### Instabilidade de formação

- Taxas de aprendizagem elevadas: Se a taxa de aprendizagem for demasiado elevada, os parâmetros da política sofrem grandes actualizações em cada passo. Isto pode potencialmente levar à não obtenção do conjunto ótimo de valores. Uma solução comum é diminuir gradualmente a taxa de aprendizagem, garantindo actualizações mais pequenas e mais estáveis à medida que a formação converge.

- Exploração excessiva: Demasiada aleatoriedade (entropia) na seleção de acções impede a convergência e conduz a grandes variações na função de perda entre passos subsequentes. Para obter um processo de formação estável e convergente, equilibre a exploração com o aproveitamento.

- Escolha incorrecta do algoritmo: Algoritmos simples como o gradiente de política podem levar a uma formação instável em ambientes complexos com grandes espaços de ação e de estado. Nesses casos, recomendamos a utilização de algoritmos mais robustos como `PPO` e `Trust Region Policy Optimization (TRPO)`. Estes algoritmos evitam grandes actualizações de políticas em cada passo e podem ser mais estáveis.

- Aleatoriedade: Os algoritmos de AR são notoriamente sensíveis aos estados iniciais e à aleatoriedade inerente à seleção de acções. Quando uma execução de treino é instável, pode por vezes ser estabilizada utilizando uma semente aleatória diferente ou reinicializando a política.

# Conclusão
Neste tutorial, exploramos os princípios básicos da AR, discutimos o *Gymnasium* como um pacote de software com uma API limpa para interagir com vários ambientes AR e mostramos como escrever um programa Python para implementar um algoritmo RL simples e aplicá-lo em um ambiente *Gymnasium*.

Depois de compreender as noções básicas neste tutorial, recomendo que utilize os ambientes do *Gymnasium* para aplicar os conceitos de AR na resolução de problemas práticos, como a otimização de rotas de táxis e simulações de negociação de acções.